# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install duckdb --quiet

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [9]:
!pip install duckdb --quiet

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

march_agg = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions,
        SUM(f.gsc_clicks) AS march_clicks,
        AVG(f.gsc_avg_position) AS march_avg_position,
        MAX(CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN NULL
            ELSE DATE_DIFF('day', c.content_updated_date, f.report_date)
        END) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# same client-grouped split as w05, same random_state -- reproduces the exact same train/test clients
unique_clients = march_agg['client_hash_id'].unique()
train_clients, test_clients = train_test_split(unique_clients, test_size=0.3, random_state=42)

labeled = march_agg.merge(april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
labeled['april_clicks'] = labeled['april_clicks'].fillna(0)
labeled['declining'] = (labeled['april_clicks'] < labeled['march_clicks'] * 0.8).astype(int)

train_df = labeled[labeled['client_hash_id'].isin(train_clients)].copy()
test_df = labeled[labeled['client_hash_id'].isin(test_clients)].copy()

features = ['march_impressions', 'march_clicks', 'march_avg_position', 'days_since_update']
train_df['days_since_update'] = train_df['days_since_update'].fillna(9999)
test_df['days_since_update'] = test_df['days_since_update'].fillna(9999)

X_train, y_train = train_df[features], train_df['declining']
X_test, y_test = test_df[features], test_df['declining']

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

print(f"Train pages: {len(train_df)}, Test pages: {len(test_df)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train pages: 116556, Test pages: 60182


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [3]:
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

march_agg = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions,
        SUM(f.gsc_clicks) AS march_clicks,
        AVG(f.gsc_avg_position) AS march_avg_position,
        MAX(CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN NULL
            ELSE DATE_DIFF('day', c.content_updated_date, f.report_date)
        END) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

labeled = march_agg.merge(april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
labeled['april_clicks'] = labeled['april_clicks'].fillna(0)
labeled['declining'] = (labeled['april_clicks'] < labeled['march_clicks'] * 0.8).astype(int)
labeled['days_since_update'] = labeled['days_since_update'].fillna(9999)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
known_staleness = labeled[labeled['days_since_update'] != 9999]
stale_threshold = known_staleness['days_since_update'].quantile(0.75)

print(f"Pages with unknown update date: {(labeled['days_since_update'] == 9999).sum()} of {len(labeled)}")
print(f"Stale threshold (based on known dates only): {stale_threshold:.0f} days")

Pages with unknown update date: 148852 of 176738
Stale threshold (based on known dates only): 34 days


In [7]:
def reason_code(row):
    good_position = row['march_avg_position'] <= good_position_threshold
    high_visibility = row['march_impressions'] >= high_visibility_threshold

    if row['declining'] == 1:
        if good_position and high_visibility:
            base = "Flagged declining despite ranking well and getting real traffic — check for an external cause (seasonality, SERP change, competitor)."
        elif good_position and not high_visibility:
            base = "Flagged declining with a good rank but low visibility — may just be losing an already-thin trickle of traffic."
        elif not good_position and high_visibility:
            base = "Flagged declining while still shown a lot — clicks appear to be drying up even as impressions hold."
        else:
            base = "Flagged declining with weak position and low visibility already — lowest-priority decline, may just be noise."
    else:
        if good_position and high_visibility:
            base = "Not flagged declining — ranks well and gets real traffic, currently stable."
        elif good_position and not high_visibility:
            base = "Not flagged declining — ranks well but barely seen, stable for now."
        elif not good_position and high_visibility:
            base = "Not flagged declining — shown a lot without ranking well, but holding steady."
        else:
            base = "Not flagged declining — low visibility and weak position, but no drop detected."

    if row['days_since_update'] != 9999:
        if row['days_since_update'] >= stale_threshold:
            base += f" Also hasn't been updated in {row['days_since_update']:.0f} days."
    else:
        base += " (Update history unknown.)"

    return base

labeled['reason_code'] = labeled.apply(reason_code, axis=1)
labeled[['content_hash_id', 'declining', 'march_avg_position', 'march_impressions', 'reason_code']].head(10)

,content_hash_id,declining,march_avg_position,march_impressions,reason_code
0,content_b7e512995f79d5a6,0,4.394234,1140.0,Not flagged declining — ranks well and gets re...
1,content_05597932fe4da067,0,2.714744,57.0,Not flagged declining — ranks well but barely ...
2,content_905aa32a0230694e,0,6.481453,149.0,Not flagged declining — low visibility and wea...
3,content_05434271b257bb68,0,6.320337,1421.0,Not flagged declining — shown a lot without ra...
4,content_d056587ff7faca0c,1,4.459107,2770.0,Flagged declining despite ranking well and get...
5,content_bfd1e41c2af250c8,0,14.753175,48.0,Not flagged declining — low visibility and wea...
6,content_2662845f598544ef,1,6.341880,150.0,Flagged declining with weak position and low v...
7,content_22610b0934f8825e,0,12.791667,67.0,Not flagged declining — low visibility and wea...
8,content_712c365258cee05c,0,4.950311,6048.0,Not flagged declining — ranks well and gets re...
9,content_476c37c366920c1b,0,50.390299,223.0,Not flagged declining — low visibility and wea...


In [10]:
test_df['declining_probability'] = model_scores
test_df['expected_clicks_at_risk'] = test_df['declining_probability'] * test_df['march_clicks']

# re-run your reason_code and action logic on test_df, since that's now your working table
test_df['reason_code'] = test_df.apply(reason_code, axis=1)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.